In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Go to Runtime -> Change runtime type -> GPU.')


Mounted at /content/drive
GPU available: True
GPU: Tesla T4
Memory: 15.6 GB


In [3]:
import json
from pathlib import Path

result_path = Path('/content/drive/MyDrive/infra_fm/results/fm_eval_satlas_multisector_v1/satlas_s2_full7region_v1_linear_probe_capped_results.json')

with open(result_path) as f:
    results = json.load(f)

# Headline
test = results['linear_probe']['test']
history = results['linear_probe']['history']
print('=== HEADLINE ===')
print(f"Test macro F1:    {test['macro_f1']:.4f}")
print(f"Test accuracy:    {test['acc']:.4f}")
print(f"Best val F1:      {max(h['val_macro_f1'] for h in history):.4f}")
tail = [h['val_macro_f1'] for h in history[-5:]]
import statistics
print(f"Tail mean F1:     {statistics.mean(tail):.4f} ± {statistics.stdev(tail):.4f}")

# Dataset sizes
print('\n=== DATASET SIZES ===')
print(f"Train n:    {results.get('dataset_info', {}).get('train_n', 'check')}")
print(f"Val n:      {results.get('dataset_info', {}).get('val_n', 'check')}")
print(f"Test n:     {results.get('dataset_info', {}).get('test_n', 'check')}")
# If those keys don't exist, look at what is in results
if 'dataset_info' not in results:
    print(f"Top-level result keys: {list(results.keys())}")
    print(f"Linear probe keys: {list(results['linear_probe'].keys())}")

# Per-class F1 + test n
print('\n=== PER-CLASS (sorted by F1) ===')
CLASS_NAMES = [
    'energy.transmission.substation', 'energy.distribution.substation',
    'energy.distribution.other', 'energy.generation.power_plant',
    'energy.generation.solar_farm', 'energy.generation.wind_farm',
    'water.wastewater.plant', 'water.treatment.plant', 'water.storage_tank',
    'transport.airport', 'transport.train_station', 'transport.port_terminal',
    'telecom.data_center'
]
per_class = list(zip(CLASS_NAMES, test['per_class_f1']))
per_class.sort(key=lambda x: -x[1])
# Test n per class from confusion matrix row totals
import numpy as np
cm = np.array(test['confusion'])
row_n = cm.sum(axis=1)
print(f"{'Class':<40s} {'F1':>6s} {'Test n':>8s}")
for i, (name, f1) in enumerate(per_class):
    idx = CLASS_NAMES.index(name)
    print(f"{name:<40s} {f1:>6.4f} {int(row_n[idx]):>8d}")

# Per-sector
print('\n=== PER-SECTOR ===')
if 'per_sector' in test:
    for sector in sorted(test['per_sector'].keys()):
        s = test['per_sector'][sector]
        print(f"  {sector:<10s} n={s['n']:>5d}  F1={s['macro_f1']:.4f}  acc={s['acc']:.4f}")

# Per-region
print('\n=== PER-REGION ===')
if 'per_region' in test:
    for region in sorted(test['per_region'].keys()):
        r = test['per_region'][region]
        print(f"  {region:<22s} n={r['n']:>5d}  F1={r['macro_f1']:.4f}  acc={r['acc']:.4f}")

=== HEADLINE ===
Test macro F1:    0.1981
Test accuracy:    0.2504
Best val F1:      0.2202
Tail mean F1:     0.1816 ± 0.0295

=== DATASET SIZES ===
Train n:    check
Val n:      check
Test n:     check
Top-level result keys: ['linear_probe']
Linear probe keys: ['run_name', 'backbone', 'condition', 'num_epochs', 'best_val_f1', 'tail_mean_f1', 'tail_std_f1', 'history', 'test']

=== PER-CLASS (sorted by F1) ===
Class                                        F1   Test n
transport.airport                        0.5412      138
water.storage_tank                       0.4403      607
transport.port_terminal                  0.4000        6
telecom.data_center                      0.2941       87
energy.generation.solar_farm             0.2489      102
water.wastewater.plant                   0.2343      202
transport.train_station                  0.2144      765
energy.distribution.substation           0.0920      152
water.treatment.plant                    0.0784      141
energy.generation

In [4]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Load the S1 result JSON
result_path = Path('/content/drive/MyDrive/infra_fm/results/fm_eval_satlas_s1_v1/satlas_s1_v1_linear_probe_results.json')
with open(result_path) as f:
    results = json.load(f)

# IMPORTANT: this confusion matrix uses the FINAL EPOCH model state.
# If we want the best-checkpoint version (the F1=0.19 we reported), 
# we'd need to re-run test eval on the best checkpoint and capture the confusion matrix from that.
# Right now the JSON only has the final-epoch confusion (which gave F1=0.13).
#
# For consistency with the §4.1.4 reported S1 F1 = 0.19, we want the best-checkpoint version.
# This means we need to recompute the confusion matrix from the best-checkpoint test eval.

cm = np.array(results['linear_probe']['test']['confusion'])
print(f'Loaded confusion matrix shape: {cm.shape}')
print(f'Test F1 in this JSON: {results["linear_probe"]["test"]["macro_f1"]:.4f}')
# This will likely show 0.13, not 0.19

Loaded confusion matrix shape: (13, 13)
Test F1 in this JSON: 0.1335


In [5]:
# Is the morning's test_result still in memory?
if 'test_result' in dir():
    import numpy as np
    cm_s1 = np.array(test_result['confusion'])
    print(f'S1 confusion matrix in memory: shape {cm_s1.shape}')
    print(f'S1 test F1 from this eval: {test_result["macro_f1"]:.4f}')
    print(f'Sum of all cells (should match test n): {cm_s1.sum()}')
else:
    print('test_result not in memory — need to re-run best-checkpoint eval')

test_result not in memory — need to re-run best-checkpoint eval


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Assumes you have cm_s2 and cm_s1 loaded as numpy arrays
# cm_s2 from your S2 result JSON (the one you used for the existing figure)
# cm_s1 from this morning's best-checkpoint test eval

# Load S2 confusion matrix (for the comparison)
import json
s2_result_path = Path('/content/drive/MyDrive/infra_fm/results/fm_eval_satlas_multisector_v1/satlas_s2_full7region_v1_linear_probe_capped_results.json')
with open(s2_result_path) as f:
    s2_results = json.load(f)
cm_s2 = np.array(s2_results['linear_probe']['test']['confusion'])

# cm_s1 should be loaded from test_result (this morning's best-checkpoint eval)
# If not already loaded:
# cm_s1 = np.array(test_result['confusion'])

CLASS_NAMES_SHORT = [
    'tx_substation',
    'dx_substation',
    'dx_other',
    'power_plant',
    'solar_farm',
    'wind_farm',
    'wastewater',
    'water_works',
    'storage_tank',
    'airport',
    'train_station',
    'port_terminal',
    'data_center',
]

# Sanity check
assert cm_s2.shape == (13, 13), f'S2 shape: {cm_s2.shape}'
assert cm_s1.shape == (13, 13), f'S1 shape: {cm_s1.shape}'

# Compute overall accuracy for each
def overall_acc(cm):
    return np.trace(cm) / cm.sum()

acc_s2 = overall_acc(cm_s2)
acc_s1 = overall_acc(cm_s1)

# Row-normalize each
def row_normalize(cm):
    row_sums = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

cm_s2_norm = row_normalize(cm_s2)
cm_s1_norm = row_normalize(cm_s1)

# Create side-by-side figure
fig, axes = plt.subplots(1, 2, figsize=(22, 9))

for ax, cm, cm_norm, title, acc, n in [
    (axes[0], cm_s2, cm_s2_norm, 'Sentinel-2 (multispectral)', acc_s2, int(cm_s2.sum())),
    (axes[1], cm_s1, cm_s1_norm, 'Sentinel-1 (SAR)', acc_s1, int(cm_s1.sum())),
]:
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    
    ax.set_xticks(np.arange(13))
    ax.set_yticks(np.arange(13))
    ax.set_xticklabels(CLASS_NAMES_SHORT, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(CLASS_NAMES_SHORT, fontsize=8)
    
    for i in range(13):
        for j in range(13):
            count = cm[i, j]
            pct = cm_norm[i, j]
            color = 'white' if pct > 0.5 else 'black'
            ax.text(j, i, f'{count}\n({pct*100:.0f}%)',
                    ha='center', va='center', fontsize=6, color=color)
    
    ax.set_xlabel('Predicted class', fontsize=10)
    ax.set_ylabel('True class', fontsize=10)
    ax.set_title(f'{title}\nOverall accuracy: {acc*100:.1f}% (n={n:,})', fontsize=11)

# Shared colorbar on the right
cbar = fig.colorbar(im, ax=axes, fraction=0.04, pad=0.04)
cbar.set_label('Row-normalized confusion rate', fontsize=10)

fig.suptitle('SatlasPretrain confusion matrices — S2 vs S1 backbones (7 regions, 13 classes)', 
             fontsize=13, y=1.02)

plt.tight_layout()

# Save
out_path = Path('/content/drive/MyDrive/infra_fm/results/confusion_matrix_s2_vs_s1_sidebyside.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved to: {out_path}')

plt.show()

NameError: name 'cm_s1' is not defined

In [2]:
import torch
from pathlib import Path
from torch.utils.data import DataLoader
import pickle

# Build model
backbone = SatlasS1Backbone(freeze=True)
model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)

# Load best checkpoint
ckpt_path = Path('/content/drive/MyDrive/infra_fm/results/fm_eval_satlas_s1_v1/satlas_s1_v1_linear_probe/checkpoint_best.pt')
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded checkpoint: epoch {ckpt["epoch"]}, val_f1={ckpt["val_macro_f1"]:.4f}')

# Build test loader (assumes test_global is S1-wrapped from the S1 notebook's data cells)
test_loader = DataLoader(
    test_global, batch_size=16, shuffle=False,
    num_workers=2, collate_fn=collate, pin_memory=True
)

# Run eval
test_result_s1 = evaluate(model, test_loader, return_breakdowns=True)
print(f'\nS1 best-checkpoint test F1: {test_result_s1["macro_f1"]:.4f}')
print(f'S1 best-checkpoint test accuracy: {test_result_s1["acc"]:.4f}')

# IMPORTANT: save to disk so we don't lose it again
import json
out_path = Path('/content/drive/MyDrive/infra_fm/results/fm_eval_satlas_s1_v1/satlas_s1_v1_best_checkpoint_test_result.json')
# Convert numpy arrays to lists for JSON serialization
test_result_serializable = {
    'macro_f1': float(test_result_s1['macro_f1']),
    'acc': float(test_result_s1['acc']),
    'per_class_f1': [float(x) for x in test_result_s1['per_class_f1']],
    'confusion': test_result_s1['confusion'].tolist() if hasattr(test_result_s1['confusion'], 'tolist') else test_result_s1['confusion'],
    'per_sector': {k: {kk: (float(vv) if isinstance(vv, (int, float)) else vv) for kk, vv in v.items()} for k, v in test_result_s1['per_sector'].items()},
    'per_region': {k: {kk: (float(vv) if isinstance(vv, (int, float)) else vv) for kk, vv in v.items()} for k, v in test_result_s1['per_region'].items()},
    'source_epoch': int(ckpt['epoch']),
    'source_val_f1': float(ckpt['val_macro_f1']),
}
with open(out_path, 'w') as f:
    json.dump(test_result_serializable, f, indent=2)
print(f'\nSaved to: {out_path}')

NameError: name 'SatlasS1Backbone' is not defined

In [4]:
import pandas as pd
import numpy as np

# Adjust path to wherever your splits parquet is
splits = pd.read_parquet(r'C:\Users\j.guthrie\Downloads\RESEARCH\infra_fm\data\PIPELINE\03-v1-samples\europe_energy_v1_sample.parquet')

# Check what's in it
print(splits.columns.tolist())
print(splits.head())
print(f'\nTotal: {len(splits)}')
print(f'\nBy split: {splits["split"].value_counts().to_dict()}')

# If lat/lon are present, compute min distance between train and test
if 'lat' in splits.columns and 'lon' in splits.columns:
    from scipy.spatial import cKDTree
    train = splits[splits['split'] == 'train']
    test = splits[splits['split'] == 'test']
    
    train_coords = train[['lat', 'lon']].values
    test_coords = test[['lat', 'lon']].values
    
    tree = cKDTree(train_coords)
    distances, _ = tree.query(test_coords, k=1)
    
    # Convert degrees to approximate km (rough at low latitudes)
    distances_km = distances * 111
    
    print(f'\nMin distance from test tile to nearest train tile: {distances_km.min():.2f} km')
    print(f'Median: {np.median(distances_km):.2f} km')
    print(f'P10: {np.percentile(distances_km, 10):.2f} km')

['asset_id', 'asset_type', 'lat', 'lon', 'name', 'source']
               asset_id                      asset_type        lat        lon  \
0     osm_way_272952694  energy.distribution.substation  51.914898   4.329453   
1     osm_way_128846707  energy.distribution.substation  47.346831   0.728605   
2  osm_node_12075264354  energy.distribution.substation  52.148092  -3.405434   
3  osm_node_10994292113  energy.distribution.substation  46.447453  28.294892   
4     osm_way_169074396  energy.distribution.substation  41.995663  13.429783   

                                name         source  
0                                     osm_geofabrik  
1  Poste électrique de Saint-Avertin  osm_geofabrik  
2         Garth Rd Builth Substation  osm_geofabrik  
3                                     osm_geofabrik  
4          Avezzano Zona industriale  osm_geofabrik  

Total: 1001


KeyError: 'split'

In [1]:
!nvidia-smi

Fri Jun 26 18:11:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   77C    P0             73W /   72W |    4148MiB /  23034MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import json
import os
from pathlib import Path
from collections import Counter, defaultdict

DATA_ROOT = Path('/content/drive/MyDrive/infra_fm/data/curated_datasets')

# Discover all manifests
manifests = sorted(DATA_ROOT.glob('dataset_*/manifest.json'))
print(f'Found {len(manifests)} manifest files:')
for m in manifests:
    print(f'  {m.parent.name}')

# Aggregate stats
total_records = 0
all_years = Counter()
region_years = defaultdict(Counter)
date_format_anomalies = []

for m_path in manifests:
    region = m_path.parent.name.replace('dataset_', '').replace('_stac_v1', '')
    with open(m_path) as f:
        manifest = json.load(f)
    records = manifest['records']
    total_records += len(records)
    
    for r in records:
        d = r.get('image_date')
        if not isinstance(d, str) or len(d) < 4:
            date_format_anomalies.append({
                'region': region,
                'asset_id': r.get('asset_id'),
                'image_date': d,
            })
            continue
        try:
            year = int(d[:4])
            all_years[year] += 1
            region_years[region][year] += 1
        except ValueError:
            date_format_anomalies.append({
                'region': region,
                'asset_id': r.get('asset_id'),
                'image_date': d,
            })

print(f'\nTotal records across all manifests: {total_records}')
print(f'\nGlobal year distribution:')
for year in sorted(all_years.keys()):
    pct = 100 * all_years[year] / total_records
    print(f'  {year}: {all_years[year]:>6,} ({pct:>5.1f}%)')

print(f'\nPer-region year distribution:')
for region in sorted(region_years.keys()):
    years_in_region = region_years[region]
    total_in_region = sum(years_in_region.values())
    year_breakdown = ', '.join(
        f'{y}: {c}' for y, c in sorted(years_in_region.items())
    )
    print(f'  {region:<20s} (n={total_in_region:>5,}): {year_breakdown}')

# AlphaEarth coverage check
ALPHA_EARTH_START = 2017
ALPHA_EARTH_LATEST = 2024  # conservative — may extend to 2025 by now
out_of_range = sum(c for y, c in all_years.items() 
                   if y < ALPHA_EARTH_START or y > ALPHA_EARTH_LATEST)
print(f'\nTiles outside AlphaEarth coverage ({ALPHA_EARTH_START}-{ALPHA_EARTH_LATEST}): {out_of_range}')

if date_format_anomalies:
    print(f'\nWARNING: {len(date_format_anomalies)} records with unparseable image_date:')
    for a in date_format_anomalies[:5]:
        print(f'  {a}')

Found 0 manifest files:

Total records across all manifests: 0

Global year distribution:

Per-region year distribution:

Tiles outside AlphaEarth coverage (2017-2024): 0
